# Практика · ML Handbook гл. 40 (40 sequences)

Источник: [sequences.html](https://mlhandbook-production.up.railway.app/parts/deep-learning/sequences.html)

Теория: [../d.5.40 Последовательности RNN LSTM GRU Transformer.md](../d.5.40 Последовательности RNN LSTM GRU Transformer.md) · [README.md](./README.md)

**Handbook-код — в этом файле**. Эталон: [handbook_gl40_практика_решения.ipynb](./handbook_gl40_практика_решения.ipynb) **после** попытки.

| ☐ | Блок |\n|---|------|\n| ☐ | §40.1 RNN с нуля |\n| ☐ | §40.1 nn.RNN |\n| ☐ | §40.3–40.4 GRU/LSTM |\n| ☐ | §40.6.1 attention |\n| ☐ | §40.6.2 MHA |\n| ☐ | §40.6.4 TransformerEncoderLayer |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


---
# §40.1 · RNN-ячейка



In [ ]:
def rnn_cell(x_seq, Wx, Wh, b):
    h = torch.zeros(Wh.shape[0])
    states = []
    for x_t in x_seq:
        h = torch.tanh(Wx @ x_t + Wh @ h + b)
        states.append(h)
    return torch.stack(states)

d_in, d_hidden = 4, 8
Wx = torch.randn(d_hidden, d_in) * 0.1
Wh = torch.randn(d_hidden, d_hidden) * 0.1
b = torch.zeros(d_hidden)
seq = torch.randn(6, d_in)
states = rnn_cell(seq, Wx, Wh, b)
print('states', tuple(states.shape))


---
# §40.1 · nn.RNN (batch_first)



In [ ]:
rnn = nn.RNN(input_size=8, hidden_size=16, batch_first=True)
batch = torch.randn(4, 10, 8)
out, h = rnn(batch)
print('out', tuple(out.shape), 'h', tuple(h.shape))


---
# §40.3–40.4 · GRU / LSTM



In [ ]:
batch = torch.randn(4, 10, 8)
gru = nn.GRU(input_size=8, hidden_size=16, batch_first=True)
out_g, h_g = gru(batch)
lstm = nn.LSTM(input_size=8, hidden_size=16, num_layers=2, batch_first=True)
out_l, (h_l, c_l) = lstm(batch)
print('GRU', tuple(out_g.shape), 'LSTM h', tuple(h_l.shape), 'c', tuple(c_l.shape))


---
# §40.6.1 · scaled dot-product attention



In [ ]:
def attention(Q, K, V):
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / d_k ** 0.5
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

seq_len, d = 4, 8
x = torch.randn(seq_len, d)
Wq, Wk, Wv = (torch.randn(d, d) for _ in range(3))
out, attn = attention(x @ Wq, x @ Wk, x @ Wv)
print('out', tuple(out.shape), 'attn', tuple(attn.shape))


---
# §40.6.2 · MultiheadAttention



In [ ]:
mha = nn.MultiheadAttention(embed_dim=16, num_heads=4, batch_first=True)
x = torch.randn(1, 5, 16)
out, w = mha(x, x, x)
print('mha out', tuple(out.shape))


---
# §40.6.4 · TransformerEncoderLayer



In [ ]:
block = nn.TransformerEncoderLayer(d_model=32, nhead=4, dim_feedforward=64, batch_first=True)
seq = torch.randn(2, 10, 32)
print('block', tuple(block(seq).shape))
